# `sat_access_script` examples
<br>
This notebook shows how to use `sat_access_script.py` and `sat_access_script.R` side by side.
Note that the ODATIS-MR examples require AVISO+ credentials. This requires opening a free account [here](https://www.aviso.altimetry.fr/en/home.html).
For the examples below, these credentials have been saved in a local `.csv` file with two columns and one row, which contain the `username` and `password` for an AVISO+ account.


## Content
This notebook contains examples for:
- Loading AVISO+ credentials from a `.csv` file
- Downloading example NetCDF files
- Plotting downloaded files
- Equivalent examples in Python and R


## Expected project layout

The examples below assume the notebook is run from the repository root and that these files are present:

```text
sat_access_script.py
sat_access_script.R
aviso_plus_pswd.csv   # your local credentials file
data/                 # output folder for downloaded files
plots/                # output folder for saved figures
```

In [ ]:
from pathlib import Path
import pandas as pd

repo_dir = Path.cwd()
credentials_csv = repo_dir / "aviso_plus_pswd.csv"
data_dir = repo_dir / "data"
plots_dir = repo_dir / "plots"

data_dir.mkdir(exist_ok=True)
plots_dir.mkdir(exist_ok=True)

repo_dir, credentials_csv, data_dir, plots_dir

## Load AVISO+ credentials

### Python

Use `pd.read_csv()` and extract the first row.

In [ ]:
credentials_df = pd.read_csv(credentials_csv)

username = credentials_df.loc[0, "usrname"]
password = credentials_df.loc[0, "psswrd"]

credentials_df.head(1)

### R

Use `read.csv()` and extract the first row.

```r
credentials_csv <- "aviso_plus_pswd.csv"
aviso_plus_cred <- read.csv(credentials_csv)

username <- aviso_plus_cred$usrname[1]
password <- aviso_plus_cred$psswrd[1]
```

## Load the scripts

### Python

Import the functions directly from `sat_access_script.py`.

In [ ]:
from sat_access_script import download_nc, plot_nc

### R

Source the script so that `download_nc()` and `plot_nc()` are available in your session.

```r
source("sat_access_script.R")
```

## Available data

The `Python` and `R` scripts in this repository provide access to the **SEXTANT** and **ODATIS-MR** suite of data via the function  `download_nc()`. Note that the naming conventions used between the `Python` and `R` scripts are the same. There are just minor language specific differences.

### Products

The top level choice when downloading data is to either access the **SEXTANT** or **ODATIS-MR** suite of products. This is done by assigning the corresponding value to `dl_product` in the `download_nc()` function in either programming language.

### Variables

It is only possible to download a single variable per function call via the argument `dl_var`. This is due to the structure of the FTP and HTML databases from which the data files are accessed.

#### **SEXTANT**

Note that validation work shows that **SEXTANT** SPM values can be used as a substitute for turbidity data when necessary.

- SPM (or SPIM) : Suspended particulate matter
- CHL (or CHLA) : Chlorophyl _a_

#### **ODATIS-MR**

- SPM (or SPIM) : Suspended particulate matter
- CHL (or CHLA) : Chlorophyl _a_
- CDOM : Coloured dissolved organic matter
- RRS : Remote sensing reflectance
- T (or TUR) : Turbidity
- SST : Sea-surface temperature (only for MODIS sensor)

### Spatial range

Note that for the **ODATIS-MR** products it is possible to cut a piece of data (i.e. a bounding box) from the larger data file at the source without downloading the entire France region. This is not however possible for **SEXTANT** data. For which the entire daily file must be downloaded.

#### **SEXTANT**

longitude range : -12 to 13

latitude range : 36 to 60

#### **ODATIS-MR**

longitude range : -7.8 to 10.3
 
latitude range : 41.2 to 51.5

### Temporal range

#### **SEXTANT**

The **SEXTANT** data are noly available as daily files. Note that because this product is level 4 (L4), every file will have fully interpolated pixels with (alomost) no gaps in space or time.

Data avilability : 1998-01-01 to NRT (near-real-time)

#### **ODATIS-MR**

For the **ODATIS-MR** products the data are available at `day`, `8-day` (i.e. weekly), and `month` temporal resolution. Note that because these are level 3 (L3) data, the values are not interpolated. Meaning that there can be gaps in 

MERIS availability : 2002-06-19 to 2012-04-08
MODIS availability : 2002-07-04 to 2024-12-31
OLCI-A availability : 2016-04-26 to 2024-12-31
OLCI-B availability : 2018-05-15 to 2024-12-31

### Atmospheric corection and processing

Note that the **SEXTANT** product is a composite of the SeaWiFS, MODIS, and MERIS satellites, the `polymer` atmospheric correction, with a further processing via the `OC5` (Ifremer) algorithm.

The **ODATIS-MR** products available via AVISO+ do not provide a choice for different corrections. For all MODIS data the `nir/swir` atmospheric correction is used. For all others it is `polymer`.

## Example 1: Download SEXTANT SPM and Chl _a_ data

For `SPM` and `CHL`, the scripts default to the `SEXTANT` product when `dl_product` is not supplied.

### Python

The example `Python` code below will download one day of SPM data for the entire France region.

In [ ]:
download_nc(
    dl_var="SPM",
    dl_dates=["2024-09-01"],
    output_dir=str(data_dir / "SEXTANT"),
    overwrite=False,
)

### R

The following `R` code will download several days of Chlorophyll _a_ data.

```r
download_nc(
  dl_var = "CHLA",
  dl_dates = c("2025-09-01", "2025-09-05"),
  output_dir = "data/SEXTANT",
  overwrite = FALSE
)
```

## Example 2: Download ODATIS-MR SST data with AVISO+ credentials

The ODATIS-MR product requires `username` and `password` from your AVISO+ account. In both versions of the download script, `SST` defaults to the `MODIS` sensor if `dl_sensor` is not supplied as this is the only sensor that provides this variable.

### Python

Download a few days of SST data.

In [ ]:
download_nc(
    dl_var="SST",
    dl_dates=["2020-09-01", "2020-09-05"],
    username=username,
    password=password,
    output_dir=str(data_dir / "MODIS"),
    overwrite=False,
)

### R

To download a single day of SST data.

```r
download_nc(
  dl_var = "SST",
  dl_dates = "2014-01-01",
  username = username,
  password = password,
  output_dir = "data/MODIS",
  overwrite = FALSE
)
```

## Example 3: Download ODATIS-MR data with an explicit sensor and bounding box

### Python

In [ ]:
download_nc(
    dl_var="SPM",
    dl_dates=["2008-12-25"],
    dl_product="ODATIS-MR",
    dl_sensor="MODIS",
    dl_bbox=[3, 4, 42.5, 44],
    username=username,
    password=password,
    output_dir=str(data_dir / "MODIS"),
    overwrite=True, # Force the download of this file
)

### R

The exact same `Python` command above, but in `R`.

```r
download_nc(
  dl_var = "SPM",
  dl_dates = c("2008-12-25"),
  dl_product = "ODATIS-MR",
  dl_sensor = "MODIS",
  dl_bbox = c(3, 4, 42.5, 44),
  username = username,
  password = password,
  output_dir = "data/MODIS",
  overwrite = TRUE # Force the download
)
```

## Example 4: Download 8-day or monthly ODATIS-MR products

### Python

An example to download two weekly (8-day) SPM data files from the OLCI-A (Sentinel-3A) sensor.

In [ ]:
download_nc(
  dl_var = "SPM",
  dl_dates = ["2017-01-01", "2017-01-09"],
  dl_product = "ODATIS-MR",
  dl_sensor = "OLCI-A",
  dl_time_step = "8-day",
  username = username,
  password = password,
  output_dir = str(data_dir / "OLCI-A"),
  overwrite = False
)

Or to download a couple of monthly CDOM files from MERIS.

In [ ]:
download_nc(
  dl_var = "CDOM",
  dl_dates = ["2009-01-01", "2009-02-01"],
  dl_product = "ODATIS-MR",
  dl_sensor = "MERIS",
  dl_time_step = "monthly",
  username = username,
  password = password,
  output_dir = str(data_dir / "MERIS"),
  overwrite = False
)

### R

An example to download 8-day chlorophyll-a data files from the OLCI-A (Sentinel-3A) sensor.

```r
download_nc(
  dl_var = "CHL",
  dl_dates = c("2019-01-01", "2019-01-25"),
  dl_product = "ODATIS-MR",
  dl_sensor = "OLCI-A",
  dl_time_step = "8-day",
  username = username,
  password = password,
  output_dir = "data/OLCI-A",
  overwrite = FALSE
)
```

Or to download monthly turbidity data from the OLCI-B (Sentinel-3B) sensor

```r
download_nc(
  dl_var = "T",
  dl_dates = c("2019-01-01", "2019-03-01"),
  dl_product = "ODATIS-MR",
  dl_sensor = "OLCI-A",
  dl_time_step = "month",
  username = username,
  password = password,
  output_dir = "data/OLCI-B",
  overwrite = FALSE
)
```

## Example 5: Plot a downloaded NetCDF file

Both scripts infer the variable name from the downloaded filename and then save a plot to `output_dir`.

### Python

In [ ]:
plot_nc(
    nc_file=str(data_dir / "SEXTANT" / "20240901-EUR-L4-SPIM-ATL-v01-fv01-OI.nc"),
    bbox=[-3.5, -0.5, 44, 48],
    plot_width=5,
    plot_height=9,
    output_dir=str(plots_dir),
)

### R

```r
plot_nc(
  nc_file = "data/MODIS/L3m_20081225__FRANCE_03_MOD_SPM-G-NS_DAY_00.nc",
  bbox = c(3, 4, 42.5, 44),
  plot_width = 5,
  plot_height = 10,
  output_dir = "plots"
)
```

## Notes

- The `.csv` file containing the credentials should not be committed to version control.
- ODATIS-MR downloads require valid AVISO+ credentials.
- If a file already exists, set `overwrite=True` in Python or `overwrite = TRUE` in R to force a fresh download.
- `plot_nc()` expects files that follow the naming structure produced by `download_nc()`.